**Question:** Does Exp C2's single-split win survive walk-forward validation?

**Assumptions:** Same discipline that overturned Exp C's claim over Exp B in notebook 06.

**Method:** Expanding-window walk-forward over Exp 0, B', C and C2.

**Result:** C2 beats ELO and C in 4/4 folds and B' in 3/4, losing only in the earliest, thinnest fold.

# 09 — Walk-Forward CV on the Top 4 Models: Exp 0 / Exp B' / Exp C / Exp C2

**Motivation.** `06_confidence_intervals_cv.ipynb` ran expanding-window walk-forward CV on the
ORIGINAL four experiments (Exp 0 / A / B / C) and found the single-split "Exp C is the best
model" headline did NOT survive time-based validation: Exp C's Brier-ranking vs Exp B flipped in
3 of its 4 folds. Since then, two stronger models were built on the single 80/20 split:

- **Exp B'** (`07_xgboost_trajectory.ipynb`) — ELO-included XGBoost + 9 hand-engineered razina-
  trajectory features. Single-split Brier 0.16545 (vs Exp B's 0.16699).
- **Exp C2** (`08_gru_elo.ipynb`) — the ELO-free GRU (Exp C) with point-in-time ELO concatenated
  at the head. Single-split Brier 0.16088 (vs Exp C's 0.16278) — the single best point estimate
  in the whole project, and (unlike Exp C vs Exp B) its edge over Exp B and Exp B' WAS
  statistically significant on the single split's paired bootstrap.

**This notebook's question:** does Exp C2's single-split win hold up under the SAME walk-forward
discipline that already overturned Exp C's "beats Exp B" claim? Or is the current #1 spot, like
before, period-dependent?

**Top 4 models walk-forward tested here:** Exp 0 (ELO baseline, closed-form) / Exp B' (ELO-
included XGB + trajectory) / Exp C (ELO-free GRU) / Exp C2 (GRU + ELO-at-head).

**Fold structure — REUSED EXACTLY from `06`, not reinvented.** All 17,128 in-scope matches,
sorted chronologically, split into 5 equal blocks. 4 expanding-window folds: fold *k* trains on
blocks `1..k`, tests on block `k+1`. Cutoffs: fold 1 = 2020-07-03, fold 2 = 2021-10-24, fold 3 =
2023-03-21, fold 4 = 2025-01-19 (fold 4 is, by construction, numerically identical to the
project's main 80/20 split — verified below as a sanity anchor, same as `06`).

**Reused verbatim, not reinvented:**
- Fold construction, `carve_train` (`calib_frac=0.15`/`early_stop_frac=0.10` chronological carve),
  `fit_xgb_fold`, `TwoTowerGRU`, `fit_gru_fold` — all copied unchanged from
  `06_confidence_intervals_cv.ipynb`.
- The 9 razina-trajectory features and `EXP_B_PRIME_COLS` construction — copied unchanged from
  `07_xgboost_trajectory.ipynb` Steps 1-4 (vectorized pandas, no per-row loops; fold-independent
  by the same point-in-time reasoning `07` already established).
- `TwoTowerGRUElo` architecture (ELO concatenated at the head: `[h_p, h_o, h_p-h_o, elo_p, elo_o]`)
  — copied unchanged from `08_gru_elo.ipynb` Step 8. The only per-fold addition: `ELO_MEAN`/
  `ELO_STD` scaling constants are recomputed from EACH FOLD's own train window (never that fold's
  test) — the same train-only hygiene discipline `08` used for the single split, just correctly
  re-applied per fold (cheap: a mean/std over already-loaded columns, no reason to reuse a stale
  global constant here the way `06` justified doing for the inert `LIGAKOEF_SCALE`).

**Point-in-time note:** none of the engineered features need to be recomputed per fold — every
feature in `features.parquet`, every trajectory feature, and every per-step GRU feature is already
computed strictly as of its own row's match date (established in `02`, re-verified in `05`/`07`).
Reslicing which matches count as "fold train" vs "fold test" does not change any row's own
feature values — only the train/test row masks change per fold. The ELO scaling constants for
Exp C2's head are the one exception recomputed per fold (see above).

**Runtime safety.** This notebook's full 4-fold loop (8 GRU retrains: Exp C + Exp C2 × 4 folds,
plus 4 cheap XGBoost fits, plus 4 closed-form ELO evaluations) was validated end-to-end in a
standalone script BEFORE being assembled here — fold 1 alone (smallest fold) confirmed all 4
models train+eval in ~70s before scaling to the full loop. Total wall time for the full 4-fold
run: **~530 seconds** (run in the background to respect the tool timeout; well within budget for
"~8 GRU retrains at ~2 min each"). No bootstrap CIs here (that is `06`'s job) — walk-forward
mean±std and per-fold ranking stability are the deliverable.


In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.calibration import CalibratedClassifierCV
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss, roc_auc_score
from torch.nn.utils.rnn import pack_padded_sequence

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 200)

DATA_DIR = Path('../data')
MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

T0 = time.time()
def lap(label):
    print(f"[{time.time()-T0:7.1f}s] {label}")

def compute_metrics_dict(y_true, p):
    p_c = np.clip(p, 1e-6, 1 - 1e-6)
    out = {
        'accuracy': accuracy_score(y_true, (p >= 0.5).astype(int)),
        'brier': brier_score_loss(y_true, p),
        'log_loss': log_loss(y_true, p_c),
    }
    try:
        out['auc'] = roc_auc_score(y_true, p)
    except ValueError:
        out['auc'] = np.nan
    return out

print('Setup OK.')


## Step 1 — Load `features.parquet` (Exp 0 / Exp B' inputs)

Same time-based `Split`/`SusretId`/`elo_diff`/`experiment_b_cols` inputs as `03`/`06`/`07`.

In [ ]:
df = pd.read_parquet(DATA_DIR / 'features.parquet')
with open(DATA_DIR / 'feature_tags.json', encoding='utf-8') as f:
    TAGS = json.load(f)
EXP_B_COLS = TAGS['experiment_b_cols']
df['liga_tip'] = df['liga_tip'].map({'SM': 0, 'SF': 1}).astype(int)
df = df.sort_values(['Datum', 'SusretId'], kind='mergesort').reset_index(drop=True)
lap(f"features.parquet loaded: {df.shape}")

# Reuse the exact tuned hyperparameters from 03_model_training.ipynb (Step 4 CV) -- no
# per-fold re-tuning, same as 06's/07's walk-forward and single-split fits.
BEST_PARAMS_B = {'max_depth': 4, 'reg_alpha': 0.1, 'reg_lambda': 1.0, 'learning_rate': 0.05}
print(f"Exp B base columns: {len(EXP_B_COLS)}")


## Step 2 — 5-block / 4-fold expanding-window walk-forward split (EXACT copy of `06` cell 13)

Fold 4's train/test boundary must exactly reproduce the project's main 80/20 split (2025-01-19) —
asserted below as the same sanity anchor `06` used.

In [ ]:
match_order_all = (df[['SusretId', 'Datum']].drop_duplicates()
                   .sort_values(['Datum', 'SusretId'], kind='mergesort').reset_index(drop=True))
n_matches_all = len(match_order_all)
N_BLOCKS = 5
block_edges = np.linspace(0, n_matches_all, N_BLOCKS + 1).astype(int)
blocks = [match_order_all.iloc[block_edges[i]:block_edges[i + 1]] for i in range(N_BLOCKS)]

FOLDS = []
for k in range(1, N_BLOCKS):
    train_ids = set(pd.concat(blocks[:k])['SusretId'])
    test_ids = set(blocks[k]['SusretId'])
    cutoff_date = blocks[k]['Datum'].min()
    FOLDS.append({'fold': k, 'train_ids': train_ids, 'test_ids': test_ids, 'cutoff': cutoff_date})
    print(f"Fold {k}: train={len(train_ids):,} matches, test={len(test_ids):,} matches, "
          f"cutoff={cutoff_date.date()}")

assert FOLDS[-1]['train_ids'].union(FOLDS[-1]['test_ids']).issuperset(set(match_order_all['SusretId']))
assert len(FOLDS[-1]['train_ids']) == TAGS['n_matches_train']
assert len(FOLDS[-1]['test_ids']) == TAGS['n_matches_test']
print('OK -- fold 4 exactly reproduces the main 80/20 split boundary (same sanity anchor as 06).')


## Step 3 — Razina-trajectory features → `EXP_B_PRIME_COLS` (EXACT copy of `07` Steps 1-4)

9 point-in-time trajectory features (`razina_ago5/10/20`, `razina_slope10`, `razina_stint`,
`razina_best_ever`, `razina_current_minus_best`, `razina_promotion_signal`, `forma_slope10`),
all vectorized pandas (`groupby`+`shift`/`rolling`/`cumcount`/`cummin`, closed-form rolling-sum
OLS slope — no per-row Python loop). Fold-independent by the same point-in-time reasoning `07`
already established (see notebook intro).

In [ ]:
raw = pd.read_csv(DATA_DIR / 'matches_export.csv', parse_dates=['Datum'])
raw['Datum'] = pd.to_datetime(raw['Datum']).dt.tz_localize(None)
raw = raw.sort_values(['Datum', 'SusretId']).reset_index(drop=True)
lap(f"matches_export.csv loaded: {raw.shape}")

common_cols = ['SusretId', 'Datum', 'SezonaId', 'LigaId', 'LigaTip', 'LigaKoeficjent', 'Lokacija']
winner_rows_t = raw[common_cols + ['WinnerId', 'LoserId']].rename(
    columns={'WinnerId': 'IgracId', 'LoserId': 'OpponentId'}).assign(Won=1)
loser_rows_t = raw[common_cols + ['LoserId', 'WinnerId']].rename(
    columns={'LoserId': 'IgracId', 'WinnerId': 'OpponentId'}).assign(Won=0)
player_log_t = pd.concat([winner_rows_t, loser_rows_t], ignore_index=True)
player_log_t = player_log_t.sort_values(['IgracId', 'Datum', 'SusretId'], kind='mergesort').reset_index(drop=True)

g_t = player_log_t.groupby('IgracId', sort=False)
player_log_t['RazinaAgo5'] = g_t['LigaKoeficjent'].shift(5)
player_log_t['RazinaAgo10'] = g_t['LigaKoeficjent'].shift(10)
player_log_t['RazinaAgo20'] = g_t['LigaKoeficjent'].shift(20)

def rolling_slope(frame, t_col, y_col, window, min_periods):
    """Closed-form rolling OLS slope of y_col vs t_col within a window (rolling sums only,
    no .apply(), no per-row loop): slope = (n*Sxy - Sx*Sy) / (n*Sxx - Sx^2)."""
    t = frame[t_col].astype(float)
    y = frame[y_col].astype(float)
    ty, tt = t * y, t * t
    grp = frame['IgracId']
    Sy = y.groupby(grp).transform(lambda s: s.rolling(window, min_periods=min_periods).sum())
    St = t.groupby(grp).transform(lambda s: s.rolling(window, min_periods=min_periods).sum())
    Sty = ty.groupby(grp).transform(lambda s: s.rolling(window, min_periods=min_periods).sum())
    Stt = tt.groupby(grp).transform(lambda s: s.rolling(window, min_periods=min_periods).sum())
    n = y.groupby(grp).transform(lambda s: s.rolling(window, min_periods=min_periods).count())
    denom = (n * Stt - St ** 2)
    return (n * Sty - St * Sy) / denom.replace(0, np.nan)

player_log_t['_pos'] = g_t.cumcount()
WINDOW = 10
player_log_t['RazinaSlope10'] = rolling_slope(player_log_t, '_pos', 'LigaKoeficjent', WINDOW, WINDOW).groupby(player_log_t['IgracId']).shift(1)
player_log_t['FormaSlope10'] = rolling_slope(player_log_t, '_pos', 'Won', WINDOW, WINDOW).groupby(player_log_t['IgracId']).shift(1)

prev_razina = g_t['LigaKoeficjent'].shift(1)
player_log_t['_changed'] = (player_log_t['LigaKoeficjent'] != prev_razina).astype(int)
player_log_t['_block_id'] = player_log_t.groupby('IgracId')['_changed'].cumsum()
player_log_t['RazinaStint'] = player_log_t.groupby(['IgracId', '_block_id']).cumcount()

player_log_t['RazinaBestEver'] = player_log_t.groupby('IgracId')['LigaKoeficjent'].apply(
    lambda s: s.cummin().shift(1)).reset_index(level=0, drop=True)
player_log_t['RazinaCurrentMinusBest'] = player_log_t['LigaKoeficjent'] - player_log_t['RazinaBestEver']

recent_min5 = player_log_t.groupby('IgracId')['LigaKoeficjent'].apply(
    lambda s: s.rolling(5, min_periods=1).min().shift(1)).reset_index(level=0, drop=True)
recent_max5 = player_log_t.groupby('IgracId')['LigaKoeficjent'].apply(
    lambda s: s.rolling(5, min_periods=1).max().shift(1)).reset_index(level=0, drop=True)
promo = pd.Series(0.0, index=player_log_t.index)
promo = promo.where(~(player_log_t['LigaKoeficjent'] < recent_min5), -1)
promo = promo.where(~(player_log_t['LigaKoeficjent'] > recent_max5), 1)
promo = promo.where(recent_min5.notna(), np.nan)
player_log_t['RazinaPromotionSignal'] = promo

TRAJ_RAW_COLS = ['RazinaAgo5', 'RazinaAgo10', 'RazinaAgo20', 'RazinaSlope10', 'RazinaStint',
                  'RazinaBestEver', 'RazinaCurrentMinusBest', 'RazinaPromotionSignal', 'FormaSlope10']
lap("Trajectory features computed (vectorized, fold-independent, same as 07)")

traj = player_log_t[['SusretId', 'IgracId'] + TRAJ_RAW_COLS].copy()
own = df[['SusretId', 'PlayerId']].merge(
    traj, left_on=['SusretId', 'PlayerId'], right_on=['SusretId', 'IgracId'], how='left', validate='one_to_one'
).drop(columns=['IgracId'])
opp = df[['SusretId', 'OpponentId']].merge(
    traj, left_on=['SusretId', 'OpponentId'], right_on=['SusretId', 'IgracId'], how='left', validate='one_to_one'
).drop(columns=['IgracId'])
assert (own['SusretId'].values == df['SusretId'].values).all()
assert (opp['SusretId'].values == df['SusretId'].values).all()

TRAJ_DIFF_COLS = []
for raw_col in TRAJ_RAW_COLS:
    diff_col = f'{raw_col}_diff'
    df[diff_col] = own[raw_col].values - opp[raw_col].values
    TRAJ_DIFF_COLS.append(diff_col)

EXP_B_PRIME_COLS = EXP_B_COLS + TRAJ_DIFF_COLS
assert 'elo_diff' in EXP_B_PRIME_COLS, "Exp B' must include elo_diff!"
lap(f"Exp B' columns ready: {len(EXP_B_PRIME_COLS)} ({len(EXP_B_COLS)} base + {len(TRAJ_DIFF_COLS)} trajectory)")


## Step 4 — Shared GRU history precompute (EXACT copy of `06` cell 15)

`MAX_LEN=24` capped history, 7-dim per-step vector, debut placeholder step. Fold-independent by
construction (a row's own history is always strictly-prior matches, regardless of which fold
later treats that row as train or test) — same reasoning `06` already used. `player_elo`/
`opponent_elo` (from `features.parquet`, the same point-in-time `EloBefore` values Exp 0/B use)
are carried alongside for Exp C2's head — the per-step sequence itself stays exactly as ELO-free
as Exp C.

In [ ]:
common_cols2 = ['SusretId', 'Datum', 'SezonaId', 'LigaId', 'LigaTip', 'LigaKoeficjent', 'Lokacija']
winner_rows = raw[common_cols2 + ['WinnerId', 'LoserId', 'W_Set1', 'W_Set2', 'W_Set3', 'L_Set1', 'L_Set2', 'L_Set3']].rename(
    columns={'WinnerId': 'IgracId', 'LoserId': 'OpponentId',
             'W_Set1': 'MySet1', 'W_Set2': 'MySet2', 'W_Set3': 'MySet3',
             'L_Set1': 'OppSet1', 'L_Set2': 'OppSet2', 'L_Set3': 'OppSet3'}
).assign(Won=1)
loser_rows = raw[common_cols2 + ['LoserId', 'WinnerId', 'L_Set1', 'L_Set2', 'L_Set3', 'W_Set1', 'W_Set2', 'W_Set3']].rename(
    columns={'LoserId': 'IgracId', 'WinnerId': 'OpponentId',
             'L_Set1': 'MySet1', 'L_Set2': 'MySet2', 'L_Set3': 'MySet3',
             'W_Set1': 'OppSet1', 'W_Set2': 'OppSet2', 'W_Set3': 'OppSet3'}
).assign(Won=0)
player_log = pd.concat([winner_rows, loser_rows], ignore_index=True)
player_log['LokacijaNorm'] = player_log['Lokacija'].str.strip().str.lower()
player_log = player_log.sort_values(['IgracId', 'Datum', 'SusretId'], kind='mergesort').reset_index(drop=True)

g = player_log.groupby('IgracId', sort=False)
player_log['MatchCountBefore'] = g.cumcount()
player_log['HomeCountBefore'] = player_log.groupby(['IgracId', 'LokacijaNorm'], sort=False).cumcount()
player_log['HomeShareBefore'] = np.where(
    player_log['MatchCountBefore'] > 0,
    player_log['HomeCountBefore'] / player_log['MatchCountBefore'], np.nan)
player_log['DaysSinceLastMatch'] = player_log.groupby('IgracId')['Datum'].diff().dt.days
player_log['CareerWinRateBefore'] = g['Won'].transform(lambda s: s.shift().expanding().mean())

for c in ['MySet1', 'MySet2', 'MySet3', 'OppSet1', 'OppSet2', 'OppSet3']:
    player_log[c + '_f'] = player_log[c].fillna(0)
player_log['SetMargin'] = (
    (player_log['MySet1_f'] + player_log['MySet2_f'] + player_log['MySet3_f'])
    - (player_log['OppSet1_f'] + player_log['OppSet2_f'] + player_log['OppSet3_f']))
player_log = player_log.drop(columns=[c + '_f' for c in ['MySet1', 'MySet2', 'MySet3', 'OppSet1', 'OppSet2', 'OppSet3']])

opp_cwr = player_log[['SusretId', 'IgracId', 'CareerWinRateBefore']].rename(
    columns={'IgracId': 'OpponentId', 'CareerWinRateBefore': 'OpponentCareerWinRateBefore'})
player_log = player_log.merge(opp_cwr, on=['SusretId', 'OpponentId'], how='left', validate='many_to_one')

MAX_LEN = 24
N_FEATS = 7
SETMARGIN_SCALE = 12.0
DAYSREST_CLIP = 90.0
# Fixed at the global max across ALL folds -- documented as inert in 05 Step 15 / reused by 06.
LIGAKOEF_SCALE_GLOBAL = float(player_log['LigaKoeficjent'].max())

step_feats = np.zeros((len(player_log), N_FEATS), dtype=np.float32)
step_feats[:, 0] = player_log['Won'].values
step_feats[:, 1] = np.clip(player_log['SetMargin'].values / SETMARGIN_SCALE, -3, 3)
days_rest = player_log['DaysSinceLastMatch'].fillna(DAYSREST_CLIP).clip(upper=DAYSREST_CLIP).values
step_feats[:, 2] = days_rest / DAYSREST_CLIP
step_feats[:, 3] = player_log['LigaKoeficjent'].values / LIGAKOEF_SCALE_GLOBAL
step_feats[:, 4] = player_log['HomeShareBefore'].fillna(0.0).values
step_feats[:, 5] = player_log['OpponentCareerWinRateBefore'].fillna(0.5).values
step_feats[:, 6] = 0.0

PLACEHOLDER_VEC = np.zeros(N_FEATS, dtype=np.float32)
PLACEHOLDER_VEC[5] = 0.5
PLACEHOLDER_VEC[6] = 1.0

player_log['_gpos'] = np.arange(len(player_log))
first_idx = player_log.groupby('IgracId')['_gpos'].min().to_dict()
idx_lookup = dict(zip(zip(player_log['IgracId'], player_log['SusretId']), player_log['_gpos']))

def build_history(igrac_id, susret_id):
    gpos = idx_lookup[(igrac_id, susret_id)]
    start = first_idx[igrac_id]
    p = gpos - start
    seq = np.zeros((MAX_LEN, N_FEATS), dtype=np.float32)
    if p == 0:
        seq[0] = PLACEHOLDER_VEC
        length = 1
    else:
        lo = max(start, gpos - MAX_LEN)
        chunk = step_feats[lo:gpos]
        length = chunk.shape[0]
        seq[:length] = chunk
    return seq, length

n_rows_pl = len(player_log)
HIST_ARRAY = np.zeros((n_rows_pl, MAX_LEN, N_FEATS), dtype=np.float32)
LEN_ARRAY = np.zeros(n_rows_pl, dtype=np.int64)
igrac_arr, susret_arr = player_log['IgracId'].values, player_log['SusretId'].values
for i in range(n_rows_pl):
    HIST_ARRAY[i], LEN_ARRAY[i] = build_history(igrac_arr[i], susret_arr[i])

HIST_T = torch.from_numpy(HIST_ARRAY)
LEN_T = torch.from_numpy(LEN_ARRAY)

# sym_gru carries Label + player_elo/opponent_elo (needed by Exp C2's head) from features.parquet
sym_gru = df[['SusretId', 'Datum', 'PlayerId', 'OpponentId', 'Label', 'Perspective', 'player_elo', 'opponent_elo']].copy()
sym_gru = sym_gru.sort_values(['Datum', 'SusretId', 'Perspective'], kind='mergesort').reset_index(drop=True)
player_pos = np.array([idx_lookup[(pid, sid)] for pid, sid in zip(sym_gru['PlayerId'], sym_gru['SusretId'])])
opp_pos = np.array([idx_lookup[(oid, sid)] for oid, sid in zip(sym_gru['OpponentId'], sym_gru['SusretId'])])

lap(f"Precomputed histories for {n_rows_pl:,} player-match rows (shared across all 4 folds, Exp C and Exp C2).")
print(f'LIGAKOEF_SCALE_GLOBAL = {LIGAKOEF_SCALE_GLOBAL}')


## Step 5 — Per-fold training helpers

`carve_train` / `fit_xgb_fold` / `TwoTowerGRU` / `fit_gru_fold`: EXACT copies of `06`'s helpers.
`TwoTowerGRUElo`: EXACT copy of `08`'s architecture (ELO concatenated at the head). New:
`bucket_tensors_elo` and `fit_gru_elo_fold`, which recompute `ELO_MEAN`/`ELO_STD` from each
fold's OWN train window (`core_ids | eval_ids | calib_ids`) before scaling — never from that
fold's test rows.

In [ ]:
def carve_train(train_match_ids, all_match_order, calib_frac=0.15, early_stop_frac=0.10):
    fold_order = all_match_order[all_match_order['SusretId'].isin(train_match_ids)].reset_index(drop=True)
    n = len(fold_order)
    n_calib = max(1, int(round(n * calib_frac)))
    fit_order = fold_order.iloc[: n - n_calib].reset_index(drop=True)
    calib_ids = set(fold_order.iloc[n - n_calib:]['SusretId'])
    n_fit = len(fit_order)
    n_eval = max(1, int(round(n_fit * early_stop_frac)))
    core_ids = set(fit_order.iloc[: n_fit - n_eval]['SusretId'])
    eval_ids = set(fit_order.iloc[n_fit - n_eval:]['SusretId'])
    return core_ids, eval_ids, calib_ids


def fit_xgb_fold(cols, best_params, core_mask, eval_mask, calib_mask, test_mask, n_estimators=500,
                  isotonic_min_rows=1000):
    X_core, y_core = df.loc[core_mask, cols], df.loc[core_mask, 'Label']
    X_eval, y_eval = df.loc[eval_mask, cols], df.loc[eval_mask, 'Label']
    X_calib, y_calib = df.loc[calib_mask, cols], df.loc[calib_mask, 'Label']
    X_test, y_test = df.loc[test_mask, cols], df.loc[test_mask, 'Label']

    model = xgb.XGBClassifier(
        n_estimators=n_estimators, max_depth=best_params['max_depth'],
        learning_rate=best_params['learning_rate'], reg_alpha=best_params['reg_alpha'],
        reg_lambda=best_params['reg_lambda'], subsample=0.9, colsample_bytree=0.9,
        random_state=RANDOM_STATE, tree_method='hist', eval_metric='logloss',
        early_stopping_rounds=30, n_jobs=-1,
    )
    model.fit(X_core, y_core, eval_set=[(X_eval, y_eval)], verbose=False)

    calib_method = 'isotonic' if len(X_calib) >= isotonic_min_rows else 'sigmoid'
    calibrated = CalibratedClassifierCV(estimator=model, method=calib_method, cv='prefit')
    calibrated.fit(X_calib, y_calib)

    p_test = calibrated.predict_proba(X_test)[:, 1]
    return p_test, y_test.values


HIDDEN_SIZE, DROPOUT, WEIGHT_DECAY, LR = 24, 0.3, 1e-4, 2e-3
BATCH_SIZE, MAX_EPOCHS, PATIENCE = 512, 60, 8


class TwoTowerGRU(nn.Module):
    def __init__(self, input_size=N_FEATS, hidden_size=24, dropout=0.3, mlp_hidden=16):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers=1, batch_first=True)
        self.drop_embed = nn.Dropout(dropout)
        self.head = nn.Sequential(
            nn.Linear(hidden_size * 3, mlp_hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(mlp_hidden, 1))

    def encode(self, seq, lengths):
        packed = pack_padded_sequence(seq, lengths.clamp(min=1).cpu(), batch_first=True, enforce_sorted=False)
        _, h_n = self.gru(packed)
        return self.drop_embed(h_n[-1])

    def forward(self, player_seq, player_len, opp_seq, opp_len):
        h_p = self.encode(player_seq, player_len)
        h_o = self.encode(opp_seq, opp_len)
        combined = torch.cat([h_p, h_o, h_p - h_o], dim=1)
        return self.head(combined).squeeze(-1)


class TwoTowerGRUElo(nn.Module):
    """Identical TwoTowerGRU architecture, with 2 extra scalar inputs (elo_p_scaled,
    elo_o_scaled) concatenated into the MLP head -- the direct parallel of Exp B's elo_diff
    feature added to XGBoost. Copied unchanged from 08_gru_elo.ipynb Step 8."""
    def __init__(self, input_size=N_FEATS, hidden_size=24, dropout=0.3, mlp_hidden=16):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers=1, batch_first=True)
        self.drop_embed = nn.Dropout(dropout)
        self.head = nn.Sequential(
            nn.Linear(hidden_size * 3 + 2, mlp_hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(mlp_hidden, 1))

    def encode(self, seq, lengths):
        packed = pack_padded_sequence(seq, lengths.clamp(min=1).cpu(), batch_first=True, enforce_sorted=False)
        _, h_n = self.gru(packed)
        return self.drop_embed(h_n[-1])

    def forward(self, player_seq, player_len, opp_seq, opp_len, elo_p, elo_o):
        h_p = self.encode(player_seq, player_len)
        h_o = self.encode(opp_seq, opp_len)
        combined = torch.cat([h_p, h_o, h_p - h_o, elo_p.unsqueeze(1), elo_o.unsqueeze(1)], dim=1)
        return self.head(combined).squeeze(-1)


def bucket_tensors(mask_ids):
    mask = sym_gru['SusretId'].isin(mask_ids).values
    pp = torch.as_tensor(player_pos[mask], dtype=torch.long)
    op = torch.as_tensor(opp_pos[mask], dtype=torch.long)
    y = torch.as_tensor(sym_gru.loc[mask, 'Label'].values, dtype=torch.float32)
    return HIST_T[pp], LEN_T[pp], HIST_T[op], LEN_T[op], y


def bucket_tensors_elo(mask_ids, elo_mean, elo_std):
    mask = sym_gru['SusretId'].isin(mask_ids).values
    pp = torch.as_tensor(player_pos[mask], dtype=torch.long)
    op = torch.as_tensor(opp_pos[mask], dtype=torch.long)
    y = torch.as_tensor(sym_gru.loc[mask, 'Label'].values, dtype=torch.float32)
    ep = torch.as_tensor(((sym_gru.loc[mask, 'player_elo'].values - elo_mean) / elo_std), dtype=torch.float32)
    eo = torch.as_tensor(((sym_gru.loc[mask, 'opponent_elo'].values - elo_mean) / elo_std), dtype=torch.float32)
    return HIST_T[pp], LEN_T[pp], HIST_T[op], LEN_T[op], y, ep, eo


def _calibrate_and_score(calib_logit_np, calib_y_np, test_logit_np):
    """Temperature scaling vs isotonic regression on the calib slice; whichever wins on the
    calib slice itself is applied to TEST -- same methodology as 05/06/08."""
    T_param = torch.nn.Parameter(torch.ones(1))
    clt, cyt = torch.from_numpy(calib_logit_np), torch.from_numpy(calib_y_np)
    opt_T = torch.optim.LBFGS([T_param], lr=0.05, max_iter=100)

    def _closure():
        opt_T.zero_grad()
        loss = nn.functional.binary_cross_entropy_with_logits(clt / T_param, cyt)
        loss.backward()
        return loss
    opt_T.step(_closure)
    T_value = float(T_param.detach().clamp(min=1e-2))

    p_calib_raw = 1 / (1 + np.exp(-calib_logit_np))
    p_calib_temp = torch.sigmoid(clt / T_value).numpy()
    iso = IsotonicRegression(out_of_bounds='clip')
    iso.fit(p_calib_raw, calib_y_np)
    p_calib_iso = iso.predict(p_calib_raw)

    p_test_raw = 1 / (1 + np.exp(-test_logit_np))
    if brier_score_loss(calib_y_np, p_calib_iso) < brier_score_loss(calib_y_np, p_calib_temp):
        p_test_cal = iso.predict(p_test_raw)
    else:
        p_test_cal = torch.sigmoid(torch.from_numpy(test_logit_np) / T_value).numpy()
    return p_test_cal


def fit_gru_fold(core_ids, eval_ids, calib_ids, test_ids, seed=RANDOM_STATE):
    """Exp C -- ELO-free GRU. Exact copy of 06's fit_gru_fold."""
    core = bucket_tensors(core_ids)
    ev = bucket_tensors(eval_ids)
    calib = bucket_tensors(calib_ids)
    test = bucket_tensors(test_ids)

    torch.manual_seed(seed)
    model = TwoTowerGRU(hidden_size=HIDDEN_SIZE, dropout=DROPOUT)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    criterion = nn.BCEWithLogitsLoss()
    n_core = core[4].shape[0]
    rng_local = np.random.RandomState(seed)

    best_brier, best_state, no_improve = np.inf, None, 0
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        perm = rng_local.permutation(n_core)
        for start in range(0, n_core, BATCH_SIZE):
            idx = perm[start:start + BATCH_SIZE]
            optimizer.zero_grad()
            logit = model(core[0][idx], core[1][idx], core[2][idx], core[3][idx])
            loss = criterion(logit, core[4][idx])
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            val_logit = model(ev[0], ev[1], ev[2], ev[3])
        val_brier = brier_score_loss(ev[4].numpy(), torch.sigmoid(val_logit).numpy())
        if val_brier < best_brier - 1e-5:
            best_brier, best_state, no_improve = val_brier, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            no_improve += 1
        if no_improve >= PATIENCE:
            break
    model.load_state_dict(best_state)

    model.eval()
    with torch.no_grad():
        calib_logit = model(calib[0], calib[1], calib[2], calib[3])
        test_logit = model(test[0], test[1], test[2], test[3])
    p_test_cal = _calibrate_and_score(calib_logit.numpy(), calib[4].numpy(), test_logit.numpy())
    return p_test_cal, test[4].numpy()


def fit_gru_elo_fold(core_ids, eval_ids, calib_ids, test_ids, seed=RANDOM_STATE):
    """Exp C2 -- GRU + ELO at the head. Same architecture/hyperparams/seed as 08, retrained per
    fold. ELO scale constants recomputed from THIS FOLD's own train window (core+eval+calib),
    never this fold's test -- correct per-fold re-application of 08's train-only hygiene."""
    train_ids_for_scale = core_ids | eval_ids | calib_ids
    scale_mask = sym_gru['SusretId'].isin(train_ids_for_scale)
    elo_mean = float(pd.concat([sym_gru.loc[scale_mask, 'player_elo'], sym_gru.loc[scale_mask, 'opponent_elo']]).mean())
    elo_std = float(pd.concat([sym_gru.loc[scale_mask, 'player_elo'], sym_gru.loc[scale_mask, 'opponent_elo']]).std())

    core = bucket_tensors_elo(core_ids, elo_mean, elo_std)
    ev = bucket_tensors_elo(eval_ids, elo_mean, elo_std)
    calib = bucket_tensors_elo(calib_ids, elo_mean, elo_std)
    test = bucket_tensors_elo(test_ids, elo_mean, elo_std)

    torch.manual_seed(seed)
    model = TwoTowerGRUElo(hidden_size=HIDDEN_SIZE, dropout=DROPOUT)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    criterion = nn.BCEWithLogitsLoss()
    n_core = core[4].shape[0]
    rng_local = np.random.RandomState(seed)

    best_brier, best_state, no_improve = np.inf, None, 0
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        perm = rng_local.permutation(n_core)
        for start in range(0, n_core, BATCH_SIZE):
            idx = perm[start:start + BATCH_SIZE]
            optimizer.zero_grad()
            logit = model(core[0][idx], core[1][idx], core[2][idx], core[3][idx], core[5][idx], core[6][idx])
            loss = criterion(logit, core[4][idx])
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            val_logit = model(ev[0], ev[1], ev[2], ev[3], ev[5], ev[6])
        val_brier = brier_score_loss(ev[4].numpy(), torch.sigmoid(val_logit).numpy())
        if val_brier < best_brier - 1e-5:
            best_brier, best_state, no_improve = val_brier, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            no_improve += 1
        if no_improve >= PATIENCE:
            break
    model.load_state_dict(best_state)

    model.eval()
    with torch.no_grad():
        calib_logit = model(calib[0], calib[1], calib[2], calib[3], calib[5], calib[6])
        test_logit = model(test[0], test[1], test[2], test[3], test[5], test[6])
    p_test_cal = _calibrate_and_score(calib_logit.numpy(), calib[4].numpy(), test_logit.numpy())
    return p_test_cal, test[4].numpy()

print('Fold-training helpers defined (Exp0 formula / Exp B\' XGB / Exp C GRU / Exp C2 GRU+ELO).')


## Step 6 — Run all 4 folds (Exp 0 / Exp B' / Exp C / Exp C2 each)

Budget: XGBoost fits in under a second per fold; the two GRUs are the heavy part
(~35-200s each per fold, proportional to that fold's train-set size — fold 4 has the most data
and takes the longest). Total expected wall time for this cell: ~530s (validated in the
standalone script run first).

In [ ]:
results = []
t_all = time.time()

for fspec in FOLDS:
    k, train_ids, test_ids, cutoff = fspec['fold'], fspec['train_ids'], fspec['test_ids'], fspec['cutoff']
    t0 = time.time()
    print(f"\n=== FOLD {k} (cutoff {cutoff.date()}, train={len(train_ids):,} matches, "
          f"test={len(test_ids):,} matches) ===")

    test_mask = df['SusretId'].isin(test_ids)
    y_test = df.loc[test_mask, 'Label'].values

    p0 = (1 / (1 + 10 ** (-df.loc[test_mask, 'elo_diff'] / 400))).values
    m0 = compute_metrics_dict(y_test, p0)
    print(f"  Exp0 (ELO, no fit)      brier={m0['brier']:.5f}  acc={m0['accuracy']:.4f}")

    core_ids, eval_ids, calib_ids = carve_train(train_ids, match_order_all)
    core_mask = df['SusretId'].isin(core_ids)
    eval_mask = df['SusretId'].isin(eval_ids)
    calib_mask = df['SusretId'].isin(calib_ids)

    pB, yB = fit_xgb_fold(EXP_B_PRIME_COLS, BEST_PARAMS_B, core_mask, eval_mask, calib_mask, test_mask)
    mB = compute_metrics_dict(yB, pB)
    print(f"  ExpB' (XGB ELO+traj)    brier={mB['brier']:.5f}  acc={mB['accuracy']:.4f}  "
          f"({time.time()-t0:.1f}s elapsed)")

    pC, yC = fit_gru_fold(core_ids, eval_ids, calib_ids, test_ids)
    mC = compute_metrics_dict(yC, pC)
    print(f"  ExpC (GRU ELO-free)     brier={mC['brier']:.5f}  acc={mC['accuracy']:.4f}  "
          f"({time.time()-t0:.1f}s elapsed)")

    pC2, yC2 = fit_gru_elo_fold(core_ids, eval_ids, calib_ids, test_ids)
    mC2 = compute_metrics_dict(yC2, pC2)
    print(f"  ExpC2 (GRU + ELO)       brier={mC2['brier']:.5f}  acc={mC2['accuracy']:.4f}  "
          f"({time.time()-t0:.1f}s elapsed)")

    for m, expname in [(m0, 'Exp 0 - ELO baseline'), (mB, "Exp B' - ELO-incl XGB + trajectory"),
                        (mC, 'Exp C - ELO-free GRU'), (mC2, 'Exp C2 - GRU + ELO')]:
        results.append({'fold': k, 'experiment': expname, 'n_train_matches': len(train_ids),
                         'n_test_matches': len(test_ids), 'cutoff': str(cutoff.date()), **m})

    print(f"  Fold {k} total: {time.time()-t0:.1f}s")

walkforward_df = pd.DataFrame(results)
print(f"\nAll folds done in {time.time()-t_all:.1f}s")


## Step 7 — Per-fold table (Brier + accuracy)

In [ ]:
EXP_ORDER = ['Exp 0 - ELO baseline', "Exp B' - ELO-incl XGB + trajectory",
             'Exp C - ELO-free GRU', 'Exp C2 - GRU + ELO']

per_fold_brier = walkforward_df.pivot(index='fold', columns='experiment', values='brier')[EXP_ORDER]
per_fold_acc = walkforward_df.pivot(index='fold', columns='experiment', values='accuracy')[EXP_ORDER]

cutoffs = walkforward_df.drop_duplicates('fold').set_index('fold')['cutoff']
n_train = walkforward_df.drop_duplicates('fold').set_index('fold')['n_train_matches']

print('Per-fold Brier (lower = better):')
per_fold_brier_display = per_fold_brier.copy()
per_fold_brier_display.insert(0, 'cutoff', cutoffs)
per_fold_brier_display.insert(1, 'n_train_matches', n_train)
per_fold_brier_display.round(5)


In [ ]:
print('Per-fold accuracy (higher = better):')
per_fold_acc_display = per_fold_acc.copy()
per_fold_acc_display.insert(0, 'cutoff', cutoffs)
per_fold_acc_display.insert(1, 'n_train_matches', n_train)
per_fold_acc_display.round(4)


## Step 8 — Mean ± std across the 4 walk-forward folds

In [ ]:
summary = walkforward_df.groupby('experiment')[['accuracy', 'brier', 'log_loss', 'auc']].agg(['mean', 'std'])
summary = summary.reindex(EXP_ORDER)
summary_display = summary.round(5)
print('Mean +/- std across the 4 walk-forward folds:')
summary_display


## Step 9 — Ranking stability: per-fold winner + pairwise comparisons

The key output of this notebook. In how many of the 4 folds is each model the best? Does the GRU
(Exp C or Exp C2) beat ELO in all 4 folds? Does Exp C2 beat Exp B' consistently, or do they flip
(the same way Exp C vs Exp B flipped in `06`)? Does Exp C2 beat Exp C consistently?

In [ ]:
print('Per-fold Brier ranking (best -> worst):')
for f in sorted(walkforward_df['fold'].unique()):
    sub = walkforward_df[walkforward_df['fold'] == f].sort_values('brier')
    ranking = ' < '.join(sub['experiment'])
    cutoff = sub['cutoff'].iloc[0]
    print(f"  Fold {f} (cutoff {cutoff}): {ranking}")
    print(f"           briers: {list(sub['brier'].round(5))}")

def beats(row, a, b):
    return row[a] < row[b]

c_beats_0 = per_fold_brier.apply(lambda r: beats(r, 'Exp C - ELO-free GRU', 'Exp 0 - ELO baseline'), axis=1)
c2_beats_0 = per_fold_brier.apply(lambda r: beats(r, 'Exp C2 - GRU + ELO', 'Exp 0 - ELO baseline'), axis=1)
bprime_beats_0 = per_fold_brier.apply(lambda r: beats(r, "Exp B' - ELO-incl XGB + trajectory", 'Exp 0 - ELO baseline'), axis=1)
c2_beats_c = per_fold_brier.apply(lambda r: beats(r, 'Exp C2 - GRU + ELO', 'Exp C - ELO-free GRU'), axis=1)
c2_beats_bprime = per_fold_brier.apply(lambda r: beats(r, 'Exp C2 - GRU + ELO', "Exp B' - ELO-incl XGB + trajectory"), axis=1)
c_beats_bprime = per_fold_brier.apply(lambda r: beats(r, 'Exp C - ELO-free GRU', "Exp B' - ELO-incl XGB + trajectory"), axis=1)

print(f"\n(a) Exp C beats Exp 0 (ELO) in {c_beats_0.sum()}/4 folds: {dict(c_beats_0)}")
print(f"    Exp C2 beats Exp 0 (ELO) in {c2_beats_0.sum()}/4 folds: {dict(c2_beats_0)}")
print(f"    Exp B' beats Exp 0 (ELO) in {bprime_beats_0.sum()}/4 folds: {dict(bprime_beats_0)}")
print(f"\n(b) Exp C2 beats Exp B' in {c2_beats_bprime.sum()}/4 folds: {dict(c2_beats_bprime)}")
print(f"    Exp C beats Exp B' in {c_beats_bprime.sum()}/4 folds: {dict(c_beats_bprime)}")
print(f"\n(c) Exp C2 beats Exp C in {c2_beats_c.sum()}/4 folds: {dict(c2_beats_c)}")

winners = walkforward_df.loc[walkforward_df.groupby('fold')['brier'].idxmin(), ['fold', 'cutoff', 'experiment', 'brier']]
print('\nPer-fold winner (lowest Brier):')
winners


## Step 10 — Plot: Brier per fold, all 4 models

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
colors = {'Exp 0 - ELO baseline': '#374151', "Exp B' - ELO-incl XGB + trajectory": '#059669',
          'Exp C - ELO-free GRU': '#7c3aed', 'Exp C2 - GRU + ELO': '#dc2626'}
markers = {'Exp 0 - ELO baseline': 's', "Exp B' - ELO-incl XGB + trajectory": 'v',
           'Exp C - ELO-free GRU': 'D', 'Exp C2 - GRU + ELO': 'o'}
for exp in per_fold_brier.columns:
    ax.plot(per_fold_brier.index, per_fold_brier[exp], label=exp, color=colors[exp],
            marker=markers[exp], linewidth=2)
ax.set_xlabel('Walk-forward fold (expanding window, later fold = more train data)')
ax.set_ylabel('Test Brier score (lower = better)')
ax.set_title('Walk-forward CV: Brier score per fold -- Exp 0 / B\' / C / C2')
ax.set_xticks(per_fold_brier.index)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../models/walkforward_top4_brier_by_fold.png', dpi=120)
plt.show()


## Step 11 — Written Summary: Is there a robust single best model?

**Per-fold Brier table** (see Step 7 for exact numbers; summarized here from this run):

| Fold | Cutoff | Train matches | Exp 0 (ELO) | Exp B' | Exp C | Exp C2 |
|---|---|---|---|---|---|---|
| 1 | 2020-07-03 | 3,425 | 0.18195 | **0.17055** | 0.18555 | 0.17730 |
| 2 | 2021-10-24 | 6,851 | 0.17585 | 0.16217 | 0.16918 | **0.16046** |
| 3 | 2023-03-21 | 10,276 | 0.16675 | 0.15247 | 0.15660 | **0.14902** |
| 4 | 2025-01-19 | 13,702 | 0.17713 | 0.16545 | 0.16254 | **0.16011** |

(Fold 4 reproduces the single-split numbers from `07`/`08` exactly: Exp 0 = 0.17713, Exp B' =
0.16545, both bit-identical to the published single-split values — confirming this replication
pipeline is correct before trusting folds 1-3, same sanity-anchor discipline as `06`.)

**Mean ± std across the 4 folds** (Brier, lower=better): Exp 0 = 0.1754±0.0064, Exp B' =
0.1627±0.0076, Exp C = 0.1685±0.0125, **Exp C2 = 0.1617±0.0117 (lowest mean)**. Mean accuracy
(higher=better): Exp 0 = 0.7490, Exp B' = 0.7588, Exp C = 0.7513, **Exp C2 = 0.7615 (highest
mean)**.

**Ranking stability — the key output:**

- **(a) Does the GRU beat ELO in all 4 folds?** Exp C2 (GRU+ELO): **YES, 4/4 folds** — fully
  robust, unlike `06`'s original finding for Exp C. Exp C (ELO-free GRU) alone: only 3/4 folds —
  it loses to plain ELO in fold 1 (the earliest, smallest-training-data fold, cutoff 2020-07-03),
  the same cold-start pattern `06` already documented. Adding ELO at the head is exactly what
  rescues the GRU in that thin-data regime.
- **(b) Does Exp C2 beat Exp B' consistently, or do they flip?** Mostly consistent, with one
  flip: **Exp C2 beats Exp B' in 3/4 folds (folds 2, 3, 4)**. It loses only in fold 1, where
  Exp B' (0.17055) is the single best model of all four and Exp C2 (0.17730) is second-best. This
  is the SAME early/thin-data fold where the ELO-free GRU (Exp C) also struggled most — the
  tabular XGBoost model is simply more sample-efficient than either GRU variant when training
  data is scarce.
- **(c) Does Exp C2 beat Exp C consistently?** **YES, 4/4 folds, with no exceptions** — adding
  point-in-time ELO at the head never hurts and always helps the GRU's point estimate, in every
  single time period tested, including the fold where the ELO-free GRU underperforms plain ELO.
  This is the walk-forward confirmation of `08`'s single-split finding (ELO is not redundant in
  the sense of being harmful) — though `08` also found the point-estimate gain over Exp C was NOT
  statistically significant on the single 80/20 split; this notebook did not re-run that
  significance test per fold (out of scope, see intro), so "wins every fold on point estimate"
  should not be read as "each individual win is statistically proven."

**Per-fold winner:** fold 1 → Exp B'; folds 2, 3, 4 → Exp C2. Exp C (ELO-free GRU alone) never
wins a single fold outright once Exp B' and Exp C2 are both in the running — it is consistently
squeezed between them, exactly the more cautious picture `06` already surfaced for the original
four experiments (Exp C's "beats Exp B" claim did not survive time-based validation; here it does
even less well against Exp B' and Exp C2, its two stronger successors built specifically to close
its ELO-free / ELO-included gaps).

**Honest bottom line.** Unlike `06`'s finding for Exp C (whose "beats Exp B" claim flipped in 3
of 4 folds and did not survive walk-forward validation), **Exp C2's advantage over Exp 0 (ELO) and
over Exp C (ELO-free GRU) IS robust across all 4 independent time folds tested here** — a
meaningfully stronger result than the original notebook `06` produced for the single-split
"winner." The one genuine instability that remains is **Exp C2 vs Exp B'**: Exp C2 wins on 3 of 4
folds (by a clear, fold-consistent margin in folds 2-4), but **in the earliest/smallest-data fold,
Exp B' is the best model of all four**, and by a wider margin than Exp C2 trails ELO. This points
at the same conclusion `07`/`08` already flagged for the single split, now confirmed across time:
**a well-tuned, hand-engineered tabular model (Exp B') is more sample-efficient early in a
league's history, while the GRU's advantage (with or without ELO) accumulates and strengthens as
more sequence data becomes available** — consistent with the general expectation that recurrent /
learned-representation models need a longer runway of training data than tree ensembles with
carefully hand-built features. If a production decision were being made today with the FULL
current dataset (the fold-4 regime, which is what the project actually has), **Exp C2 is the most
robust and highest-point-estimate choice** — not just on the single 80/20 split (where `08` also
found its edge over Exp B/Exp B' to be statistically significant) but now also across 3 independent
prior time periods. But "Exp C2 is the single best model, full stop, regardless of how much
history is available" is NOT supported — the correct, walk-forward-validated statement is: **Exp
C2 is the best model once the league has accumulated a few years of match history (folds 2-4);
Exp B' is competitive-to-better in the earliest, thinnest-data regime (fold 1).** This is a more
nuanced and better-supported conclusion than either single-split result alone could offer, and it
is exactly the kind of period-dependency the walk-forward check was built to catch — the same
spirit as `06`'s original caution, but this time landing on a more encouraging (partially robust,
not merely "tied") answer for the newest model in the lineup.

**Caveats (same scope limits as `06`/`07`/`08`):** no bootstrap CIs per fold here (by design —
`06` already established the bootstrap methodology; re-running it 4× more
would be redundant without adding walk-forward-specific insight and would blow the runtime
budget). NN training carries residual seed-to-seed stochasticity (not re-verified with a
multi-seed sweep here, same caveat `05`/`08` already flagged). Only 4 folds, each itself only
~3,425 test matches — the fold-1 "Exp B' wins" result, while clean, rests on the single smallest
and noisiest test slice of the four.


**This is the project's final word.** Notebooks 01–04 established the baseline three-experiment comparison (Exp 0/A/B) and its single-split null result for ELO-free XGBoost; 05 found a seemingly dominant ELO-free GRU (Exp C); 06 showed that single-split "beats everything" claim does not survive bootstrap/walk-forward validation; 07 traced Exp C's edge to razina-trajectory and showed hand-engineered XGBoost (Exp A'/B') can recover most, not all, of it; 08 added ELO to the GRU (Exp C2) and found it the best point estimate in the project; this notebook is where all of that gets tested against time. See `../README.md` for the full seven-experiment results table and the complete narrative arc from Exp 0 through Exp C2.


---

**Update (see `12_walkforward_with_c3_c4.ipynb`):** this notebook's period-dependency finding (Exp B' wins the earliest, thinnest fold; Exp C2 wins folds 2-4) was established for the PRE-enrichment model lineup only. Two further-enriched GRU variants built afterward (Exp C3/C4, `10_enriched_gru.ipynb`/`11_enriched_gru_text.ipynb`) close that fold-1 gap entirely — the enriched family beats Exp B' in all 4 folds, including fold 1 — so this notebook's caution that Exp C2 is not "the single best model, full stop" no longer applies to the improved C3/C4 successors, even though it remains the correctly-scoped word for Exp C2 considered on its own.
